In [1]:
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
import re
from spc import SPC3
from qua_tools_nv2.dataset import DatasetReader


In [5]:
spad_dir = r'C:\Users\SPUD1\Documents\experiment_workspace\SPAD data'
qm_dir = r'C:\Users\SPUD1\Documents\experiment_workspace\qua-libs\Quantum-Control-Applications\Optically addressable spin qubits\NV2_array_SPAD'

spad_file_header = 'contacq_ODMR_test'
qm_file    = r'2026-03-11\#73_pulsed_odmr_203241'

In [6]:
# Toggle whether to use LO frequency (absolute MW freq) or just IF
USE_LO_FREQUENCY = False  # Set False to work in IF space

reader = DatasetReader(qm_dir)

ds_folder = reader.resolve_dataset(qm_file).folder
ds = reader.resolve_dataset(ds_folder)
data = reader.load(ds)

if_hz = np.asarray(data['IF_frequencies'], dtype=float)

cfg = data.get('config') or {}
if USE_LO_FREQUENCY:
    lo_hz = float(cfg['elements']['NV']['mixInputs']['lo_frequency'])
    f_mw_hz = lo_hz + if_hz
else:
    lo_hz = None
    f_mw_hz = if_hz

n_avg = int(data.get('n_avg', 1))
# 'iteration' is 0-indexed:
# Falls back to n_avg when the field is absent
n_iteration = int(data.get('iteration', n_avg - 1)) + 1

print(f'Loaded: {ds.folder}')
print(f"n_avg        : {n_avg}  (requested)")
print(f"n_iteration  : {n_iteration}  (actually completed)")
if n_iteration != n_avg:
    print(f"  ↳ experiment was interrupted at {n_iteration / n_avg * 100:.1f}% of target")
print(f"len(if_hz)   : {len(if_hz)}")
print(f"if_hz range  : {if_hz.min()/1e6:.1f} – {if_hz.max()/1e6:.1f} MHz")
print(f"Expected total frames: {2 * n_iteration * len(if_hz)}") #sig,ref frames per iteration, per frequency point


Loaded: C:\Users\SPUD1\Documents\experiment_workspace\qua-libs\Quantum-Control-Applications\Optically addressable spin qubits\NV2_array_SPAD\Data\2026-03-11\#73_pulsed_odmr_203241
n_avg        : 500000  (requested)
n_iteration  : 500000  (actually completed)
len(if_hz)   : 40
if_hz range  : 70.0 – 89.5 MHz
Expected total frames: 40000000


In [7]:

spad_dir_path = Path(spad_dir)

# -----------------------------------------------------------------
# Discover all part-files matching <header>.spc3, <header>2.spc3, ...
# Only digits (never other text) may follow the header — this prevents
# "ODMR_test_new.spc3" being picked up when header is "ODMR_test".
# -----------------------------------------------------------------
def _spc3_sort_key(p):
    m = re.search(r'(\d+)\.spc3$', p.name, re.IGNORECASE)
    return int(m.group(1)) if m else 0

_part_re = re.compile(
    r'^' + re.escape(spad_file_header) + r'\d*\.spc3$',
    re.IGNORECASE
)

spc3_files = sorted(
    (p for p in spad_dir_path.glob(f'{spad_file_header}*.spc3') if _part_re.match(p.name)),
    key=_spc3_sort_key
)

if not spc3_files:
    raise FileNotFoundError(
        f"No SPC3 files found matching '{spad_file_header}[<digits>].spc3' in\n  {spad_dir_path}"
    )

print(f"Found {len(spc3_files)} SPC3 file(s):")
for fp in spc3_files:
    print(f"  {fp.name}")


Found 39 SPC3 file(s):
  contacq_ODMR_test.spc3
  contacq_ODMR_test2.spc3
  contacq_ODMR_test3.spc3
  contacq_ODMR_test4.spc3
  contacq_ODMR_test5.spc3
  contacq_ODMR_test6.spc3
  contacq_ODMR_test7.spc3
  contacq_ODMR_test8.spc3
  contacq_ODMR_test9.spc3
  contacq_ODMR_test10.spc3
  contacq_ODMR_test11.spc3
  contacq_ODMR_test12.spc3
  contacq_ODMR_test13.spc3
  contacq_ODMR_test14.spc3
  contacq_ODMR_test15.spc3
  contacq_ODMR_test16.spc3
  contacq_ODMR_test17.spc3
  contacq_ODMR_test18.spc3
  contacq_ODMR_test19.spc3
  contacq_ODMR_test20.spc3
  contacq_ODMR_test21.spc3
  contacq_ODMR_test22.spc3
  contacq_ODMR_test23.spc3
  contacq_ODMR_test24.spc3
  contacq_ODMR_test25.spc3
  contacq_ODMR_test26.spc3
  contacq_ODMR_test27.spc3
  contacq_ODMR_test28.spc3
  contacq_ODMR_test29.spc3
  contacq_ODMR_test30.spc3
  contacq_ODMR_test31.spc3
  contacq_ODMR_test32.spc3
  contacq_ODMR_test33.spc3
  contacq_ODMR_test34.spc3
  contacq_ODMR_test35.spc3
  contacq_ODMR_test36.spc3
  contacq_ODMR_

In [ ]:
# extract metadata from first file
frames, header = SPC3.ReadSPC3DataFile(spc3_files[0])

# frames shape: (n_counters, n_frames, n_cols, n_rows)
print("=== Header (first file) ===")
#print(f"  camera_id    : {header.camera_id}")
#print(f"  serial       : {header.SN}")
#print(f"  firmware ver : {header.FW_VER}")
print(f"  n_frames     : {header.N_frames}")
print(f"  n_counters   : {header.N_counters}")
print(f"  n_rows       : {header.N_rows}")
print(f"  n_cols       : {header.N_cols}")
print(f"  bits/pixel   : {header.bit_x_pix}")
print(f"  n_pixels     : {header.N_pix}")
print(f"  HW integ time: {header.HwIntTime * 1e6:.3f} µs")
print(f"  summed frames: {header.SummedFrames}")
print(f"  coarse gate C1: {header.CoarseGate_C1_ON}  "
      f"({header.CoarseGate_C1_startPos * 1e9:.0f} – {header.CoarseGate_C1_stopPos * 1e9:.0f} ns)")

print(f"\n=== Frames (first file) ===")
print(f"  shape : {frames.shape}  (n_counters, n_frames, n_cols, n_rows)")
print(f"  dtype : {frames.dtype}")
print(f"  min/max (counter 0): {frames[0].min()}, {frames[0].max()}")


=== Header ===
  n_frames     : 1048574
  n_counters   : 1
  n_rows       : 32
  n_cols       : 32
  bits/pixel   : 8
  n_pixels     : 1024
  HW integ time: 10.400 µs
  summed frames: 1
  coarse gate C1: False  (0 – 0 ns)

=== Frames ===
  shape : (1, 1048574, 32, 32)  (n_counters, n_frames, n_cols, n_rows)
  dtype : uint8
  min/max (counter 0): 0, 3


In [9]:
F              = len(if_hz)
frames_per_rep = 2 * F

header          = None
sig_sum         = None   # (F, rows, cols) float64
ref_sum         = None   # (F, rows, cols) float64
n_complete_reps = 0
n_rows = n_cols = None
carry           = None
N_frames_raw    = 0


for fp in spc3_files:
    f_data, h = SPC3.ReadSPC3DataFile(str(fp))
    frames = f_data[0]          # (N, rows, cols) — native uint8 or uint16
    N_frames_raw += frames.shape[0]

    if header is None:
        header = h
        n_rows, n_cols = frames.shape[1], frames.shape[2]
        sig_sum = np.zeros((F, n_rows, n_cols), dtype=np.float64)
        ref_sum = np.zeros((F, n_rows, n_cols), dtype=np.float64)

    if carry is not None:
        frames = np.concatenate([carry, frames], axis=0)
        carry = None

    n_frames        = frames.shape[0]
    n_complete_here = (n_frames // frames_per_rep) * frames_per_rep

    if n_complete_here > 0:
        batch = frames[:n_complete_here].astype(np.float64)

        rep_view = batch.reshape(-1, F, 2, n_rows, n_cols)
        sig_sum += rep_view[:, :, 0, :, :].sum(axis=0)
        ref_sum += rep_view[:, :, 1, :, :].sum(axis=0)
        n_complete_reps += n_complete_here // frames_per_rep

    if n_frames > n_complete_here:
        carry = frames[n_complete_here:]

    print(f"  {fp.name}: {n_complete_here} frames used  "
          f"(running total: {n_complete_reps} complete reps)")

dropped = N_frames_raw - n_complete_reps * frames_per_rep
print(f"\nTotal frames raw     : {N_frames_raw}")
print(f"Complete repetitions : {n_complete_reps}  ({n_complete_reps / n_iteration * 100:.1f}% of {n_iteration} completed, {n_avg} requested)")
if dropped:
    print(f"Dropped trailing     : {dropped} frame(s) (incomplete final repetition)")



  contacq_ODMR_test.spc3: 1048560 frames used  (running total: 13107 complete reps)
  contacq_ODMR_test2.spc3: 1048560 frames used  (running total: 26214 complete reps)
  contacq_ODMR_test3.spc3: 1048560 frames used  (running total: 39321 complete reps)
  contacq_ODMR_test4.spc3: 1048560 frames used  (running total: 52428 complete reps)
  contacq_ODMR_test5.spc3: 1048560 frames used  (running total: 65535 complete reps)
  contacq_ODMR_test6.spc3: 1048640 frames used  (running total: 78643 complete reps)
  contacq_ODMR_test7.spc3: 1048560 frames used  (running total: 91750 complete reps)
  contacq_ODMR_test8.spc3: 1048560 frames used  (running total: 104857 complete reps)
  contacq_ODMR_test9.spc3: 1048560 frames used  (running total: 117964 complete reps)
  contacq_ODMR_test10.spc3: 1048560 frames used  (running total: 131071 complete reps)
  contacq_ODMR_test11.spc3: 1048560 frames used  (running total: 144178 complete reps)
  contacq_ODMR_test12.spc3: 1048640 frames used  (running to